In [29]:
import pandas as pd
import polars as pl

from ds_practice.bench import bench_table

pl.Config.set_tbl_rows(50)  # show more rows

polars.config.Config

In [2]:
PATH = "../data/yellow_tripdata_2026-08.parquet"
lf = pl.scan_parquet(PATH)
print(type(lf))
schema = lf.collect_schema()
print(len(schema), list(schema)[:4])
lf.select(pl.len()).collect()

<class 'polars.lazyframe.frame.LazyFrame'>
21 ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count']


len
u32
3336716


In [3]:
documented = {
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "RatecodeID",
    "store_and_fwd_flag",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "Airport_fee",
    "cbd_congestion_fee",
}
actual = set(schema)
print("extra in file:   ", actual - documented)
print("missing in file: ", documented - actual)
print(schema)  # full names + dtypes

extra in file:    {'request_source'}
missing in file:  set()
Schema({'VendorID': Int32, 'tpep_pickup_datetime': Datetime(time_unit='us', time_zone=None), 'tpep_dropoff_datetime': Datetime(time_unit='us', time_zone=None), 'passenger_count': Int64, 'trip_distance': Float64, 'RatecodeID': Int64, 'store_and_fwd_flag': String, 'PULocationID': Int32, 'DOLocationID': Int32, 'payment_type': Int64, 'fare_amount': Float64, 'extra': Float64, 'mta_tax': Float64, 'tip_amount': Float64, 'tolls_amount': Float64, 'improvement_surcharge': Float64, 'total_amount': Float64, 'congestion_surcharge': Float64, 'Airport_fee': Float64, 'cbd_congestion_fee': Float64, 'request_source': String})


In [8]:
lf.select("request_source").describe()

statistic,request_source
str,str
"""count""","""920849"""
"""null_count""","""2415867"""
"""mean""",null
"""std""",null
"""min""","""A"""
"""25%""",null
"""50%""",null
"""75%""",null
"""max""","""HV0005"""


In [9]:
quality = lf.select(
    # comparison on col gives bool Series (1/0), .sum() counts Trues
    (pl.col("fare_amount") < 0).sum().alias("neg_fare_count"),
    # .mean() on bool = prop of Trues (sum / count)
    (pl.col("trip_distance") == 0).mean().alias("zero_distance_share"),
    # .quantile(q) where q is b/w 0.0 and 1.0, 99.9th = 0.999
    pl.col("trip_distance").quantile(0.999).alias("p999_trip_distance"),
    # .null_count() counts None/null values in col
    pl.col("passenger_count").null_count().alias("passenger_count_nulls"),
).collect()
quality

neg_fare_count,zero_distance_share,p999_trip_distance,passenger_count_nulls
u32,f64,f64,u32
14049,0.035681,32.82,921181


In [11]:
# polars will re-order in cheapest way to run
# bottom 4 lines is comparison (where the data comes in)
q = (
    lf.filter(pl.col("trip_distance") > 0, pl.col("fare_amount") > 0)
    .with_columns(hour=pl.col("tpep_pickup_datetime").dt.hour())
    .group_by("hour")
    .agg(trips=pl.len(), median_fare=pl.col("fare_amount").median())
    .sort("hour")
)
print(q.explain(optimized=False))
print("-" * 60)
print(q.explain())
q.collect()

SORT BY [col("hour")]
  AGGREGATE[maintain_order: false]
    [len().alias("trips"), col("fare_amount").median().alias("median_fare")] BY [col("hour")]
    FROM
     WITH_COLUMNS:
     [col("tpep_pickup_datetime").dt.hour().alias("hour")] 
      FILTER col("fare_amount") > 0.0
      FROM
        FILTER col("trip_distance") > 0.0
        FROM
          Parquet SCAN [../data/yellow_tripdata_2026-08.parquet]
          PROJECT */21 COLUMNS
          ESTIMATED ROWS: 3336716
------------------------------------------------------------
SORT BY [col("hour")]
  AGGREGATE[maintain_order: false]
    [len().alias("trips"), col("fare_amount").median().alias("median_fare")] BY [col("hour")]
    FROM
    simple π 2/2 ["hour", "fare_amount"]
       WITH_COLUMNS:
       [col("tpep_pickup_datetime").dt.hour().alias("hour")] 
        simple π 2/2 ["tpep_pickup_datetime", ... 1 other column]
          Parquet SCAN [../data/yellow_tripdata_2026-08.parquet]
          PROJECT 3/21 COLUMNS
          SELECTION:

hour,trips,median_fare
i8,u32,f64
0,112346,18.53
1,80593,18.21
2,54716,18.4
3,38036,18.86
4,30253,20.51
…,…,…
19,186964,14.9
20,188799,15.6
21,189592,16.3


projection pushdown: pl showed that only 3 of 21 columns needed to be read
predicate pushdown: FILTER node gone, moved inside the scan as selection (instead of 'load everything, then throw rows away, polars filters while it reads). -> can skip whole chunks via the min/max values in the footer.

In [16]:
# columns: LocationID, Borough, Zone, service_zone
zones = pl.scan_csv("../data/taxi_zone_lookup.csv")

# Finding 1: demand by hour. Reuse q from Step 3.
demand_by_hour = (
    lf.filter(pl.col("trip_distance") > 0, pl.col("fare_amount") > 0)
    .with_columns(hour=pl.col("tpep_pickup_datetime").dt.hour())
    .group_by("hour")
    .agg(trips=pl.len(), median_fare=pl.col("fare_amount").median())
    .sort("trips")
)
demand_by_hour.collect()

hour,trips,median_fare
i8,u32,f64
4,30253,20.51
5,31403,19.25
3,38036,18.86
2,54716,18.4
6,55927,16.3
1,80593,18.21
7,86766,15.005
0,112346,18.53
8,120379,15.6


Demand peaks at 17:00 and 18:00 aligning with post-work commutes and dips overnight — high trust as timestamps are machine-generated with few nulls.

In [25]:
# Finding 2 TODO: average tip as % of fare, by payment_type
#   hint: keep fare_amount > 0; tip_amount / fare_amount inside .agg
#   then check the data dictionary: what's odd about cash (payment_type 2)?

# filter out any neg fare_amount
tip_perc_by_payment_type = (
    lf.filter(pl.col("fare_amount") > 0)
    .group_by("payment_type")
    .agg(
        (pl.col("tip_amount") / pl.col("fare_amount")).mean().alias("avg_tip_pct"),
        pl.len().alias("trip_count"),
    )
    .sort("payment_type")
    .with_columns(
        pl.col("payment_type")
        .cast(pl.String)
        .replace(
            {
                "0": "Unmapped",
                "1": "Credit Card",
                "2": "Cash",
                "3": "No charge",
                "4": "Dispute",
                "5": "Unkown",
                "6": "Voided trip",
            }
        )
        .alias("payment_type")
    )
    .collect()
)
tip_perc_by_payment_type

payment_type,avg_tip_pct,trip_count
str,f64,u32
"""Unmapped""",0.013653,920219
"""Credit Card""",0.252742,2051317
"""Cash""",0.000039,325430
"""No charge""",0.000363,8625
"""Dispute""",0.00011,13876


Credit card tips average 25% versus ~0% for cash. However, cash tips go directly to the driver, and aren't recorded electronically, thus the result reflects a data collection gap, and is not reflective of actual tipping behaviour; disputes and no-charge tips show near-zero tips, meeting expectation.

In [18]:
# Finding 3 TODO: 5 busiest pickup zones with their borough
#   hint: .join(zones, left_on=..., right_on=...), then .top_k(5, by=...)

top_5_pickup_zones = (
    lf.group_by("PULocationID")
    .agg(pl.len().alias("trip_count"))
    .join(zones, left_on="PULocationID", right_on="LocationID")
    .top_k(5, by="trip_count")
    .collect()
)
top_5_pickup_zones

PULocationID,trip_count,Borough,Zone,service_zone
i32,u32,str,str,str
132,152581,"""Queens""","""JFK Airport""","""Airports"""
161,136671,"""Manhattan""","""Midtown Center""","""Yellow Zone"""
237,128140,"""Manhattan""","""Upper East Side South""","""Yellow Zone"""
186,116496,"""Manhattan""","""Penn Station/Madison Sq West""","""Yellow Zone"""
236,105366,"""Manhattan""","""Upper East Side North""","""Yellow Zone"""


The 5 busiest zones are JFK Airport, Midtown Center, Upper East Side South, Penn Station/Madison Sq West, and Upper East Side North, concentrated in Manhattan and airports - high trust as PULocationID is a required meter field with very few nulls.

In [ ]:
# Finding 4 TODO: trips by weekday × hour
#   hint: .dt.weekday() is 1 = Mon ... 7 = Sun; group_by both, .len(),
#         collect, THEN .pivot(on="hour", index=..., values="len")

trips_grid = (
    lf.with_columns(
        pl.col("tpep_pickup_datetime").dt.weekday().alias("weekday"),
        pl.col("tpep_pickup_datetime").dt.hour().alias("hour"),
    )
    .group_by("weekday", "hour")
    .len()
    .collect()
    .pivot(on="hour", index="weekday", values="len")
    .sort("weekday")
)
sorted_cols = ["weekday"] + [str(h) for h in range(24)]
trips_grid = trips_grid.select(sorted_cols)
trips_grid

weekday,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
i8,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
1,10110,5382,2807,2007,2710,5762,11327,18615,24741,24493,22236,22473,23005,24688,27740,27287,28050,34013,33214,26743,25328,25371,19802,14348
2,6794,3625,1810,1273,1712,3899,8593,14378,20671,21198,19956,20100,20958,22246,23745,24132,24682,28020,29411,24903,24758,27688,23040,15961
3,8517,4387,2275,1568,1827,3690,7831,13789,20302,20512,19772,20561,22035,22494,24787,25175,26365,30832,32952,28312,26270,28430,24979,18530
4,10493,5576,2965,2070,2265,4068,8234,14142,20649,20934,20388,20992,22114,24829,26191,26413,29445,36795,38931,34204,30705,30796,29099,22502
5,14100,8952,5650,3547,3027,4458,7903,11423,15100,16618,17606,18651,20648,22117,24422,25538,26557,28613,35082,31062,26117,23942,25936,25983
6,30816,26126,19336,13676,8913,4860,6685,8424,11160,16620,20474,24002,26877,29176,28904,31627,33627,35403,37719,34778,31857,31259,35222,36682
7,33752,28336,21284,14970,10738,5582,6463,7464,9727,14627,19035,23234,26036,28373,29537,30302,30046,29688,29360,26301,26620,24853,20870,15446


Weekday trips cluster around commute hours while weekends spread more evenly across the day - high trust as pickup timestamps are machine-generated with low null rates.

In [28]:
# Finding 5 TODO: your choice, e.g. trip duration outliers:
#   hint: (dropoff - pickup) gives a Duration; .dt.total_minutes()

trip_duration_outliers = (
    lf.with_columns(
        duration=(
            pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime")
        ).dt.total_minutes()
    )
    .select(
        (pl.col("duration") < 0).sum().alias("neg_mins"),
        (pl.col("duration") > 180).sum().alias("over_180"),
        pl.col("duration").quantile(0.999).alias("p999_mins"),
    )
    .collect()
)
trip_duration_outliers

neg_mins,over_180,p999_mins
u32,u32,f64
3,870,89.0


3 negative-duration trips suggest minor data entry errors, 870 trips exceed 3 hours (likely forgotten meters), but 99.9% of trips finish under 89 minutes indicating no extreme tail - moderate trust, would exclude the 3 negative rows from any duration analysis.

In [30]:
def polars_lazy():
    return q.collect()


# TODO 1 polars_eager: pl.read_parquet(PATH), then the same filter/group_by
def polars_eager():
    df = pl.read_parquet(PATH)
    return (
        df.filter(pl.col("trip_distance") > 0, pl.col("fare_amount") > 0)
        .with_columns(hour=pl.col("tpep_pickup_datetime").dt.hour())
        .group_by("hour")
        .agg(trips=pl.len(), median_fare=pl.col("fare_amount").median())
        .sort("hour")
    )


# TODO 2 pandas_all: pd.read_parquet(PATH), boolean mask, groupby(...dt.hour)
#        .fare_amount.agg(["size", "median"])
def pandas_all():
    df = pd.read_parquet(PATH)
    df_filter = df[(df["trip_distance"] > 0) & (df["fare_amount"] > 0)]
    df_filter["hour"] = df_filter["tpep_pickup_datetime"].dt.hour
    return df_filter.groupby("hour")["fare_amount"].agg(["size", "median"])


# TODO 3 pandas_3_cols: same as TODO 2 but pass columns=[...] to read_parquet
def pandas_3_cols():
    df = pd.read_parquet(
        PATH, columns=["trip_distance", "fare_amount", "tpep_pickup_datetime"]
    )
    df_filter = df[(df["trip_distance"] > 0) & (df["fare_amount"] > 0)]
    df_filter["hour"] = df_filter["tpep_pickup_datetime"].dt.hour
    return df_filter.groupby("hour")["fare_amount"].agg(["size", "median"])


cases = {
    "polars_lazy": polars_lazy,
    "polars_eager": polars_eager,
    "pandas_all": pandas_all,
    "pandas_3_cols": pandas_3_cols,
}
bench_table(cases, repeats=3)

case,best_s,vs_fastest
str,f64,f64
"""polars_lazy""",0.035268,1.0
"""pandas_3_cols""",0.097399,2.761684
"""polars_eager""",0.119209,3.380088
"""pandas_all""",0.166286,4.714924


Polars lazy is 4.7x faster than pandas_all because it skips unneeded columns at read time AND Polars' multi-threaded engine outperforms pandas even when both load all data (polars_eager beats pandas_3_cols despite reading more columns).